In [1]:
# Cell 1 — imports (standard libraries only)
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import fiona
import rasterio
import matplotlib.pyplot as plt

DATA_RAW = Path("../data/raw/cempaka_indonesia")
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:

def sample_raster_at_points(raster_path, points_gdf, band=1):
    """Read pixel values at each point's location — no external module needed."""
    with rasterio.open(raster_path) as src:
        pts = points_gdf.to_crs(src.crs) if points_gdf.crs != src.crs else points_gdf
        coords = [(geom.x, geom.y) for geom in pts.geometry]
        values = np.array([v[band - 1] for v in src.sample(coords)])
    return values

In [3]:
for f in sorted(DATA_RAW.rglob("*")):
    if f.is_file():
        print(f.relative_to(DATA_RAW), f"({f.stat().st_size / 1e6:.1f} MB)")

Landslide Controlling Factors\map_aspect.tfw (0.0 MB)
Landslide Controlling Factors\map_aspect.tif (96.5 MB)
Landslide Controlling Factors\map_aspect.tif.aux.xml (0.0 MB)
Landslide Controlling Factors\map_aspect.tif.xml (0.0 MB)
Landslide Controlling Factors\map_distriv.tfw (0.0 MB)
Landslide Controlling Factors\map_distriv.tif (96.5 MB)
Landslide Controlling Factors\map_distriv.tif.aux.xml (0.0 MB)
Landslide Controlling Factors\map_distriv.tif.xml (0.0 MB)
Landslide Controlling Factors\map_distroad.tfw (0.0 MB)
Landslide Controlling Factors\map_distroad.tif (96.5 MB)
Landslide Controlling Factors\map_distroad.tif.aux.xml (0.0 MB)
Landslide Controlling Factors\map_distroad.tif.xml (0.0 MB)
Landslide Controlling Factors\map_elev.tfw (0.0 MB)
Landslide Controlling Factors\map_elev.tif (96.5 MB)
Landslide Controlling Factors\map_elev.tif.aux.xml (0.0 MB)
Landslide Controlling Factors\map_elev.tif.xml (0.0 MB)
Landslide Controlling Factors\map_geologi.tfw (0.0 MB)
Landslide Controlling Fac

In [4]:
# find the gpkg and list its layers
gpkg_files = list(DATA_RAW.rglob("*.gpkg"))
assert gpkg_files, "No .gpkg file found — check the extracted folder structure"
gpkg_path = gpkg_files[0]
print("Using:", gpkg_path)

layers = fiona.listlayers(gpkg_path)
print("Layers found:", layers)

Using: ..\data\raw\cempaka_indonesia\Landslide Point\ls_center.gpkg
Layers found: ['ls_center']


In [5]:
#inspect each layer before deciding which is which
for layer in layers:
    gdf = gpd.read_file(gpkg_path, layer=layer, rows=3)
    print(f"\n=== {layer} ===")
    print("Columns:", gdf.columns.tolist())
    print("Geometry type:", gdf.geom_type.unique())
    print("CRS:", gdf.crs)


=== ls_center ===
Columns: ['lslpts', 'X', 'Y', 'geometry']
Geometry type: ['Point']
CRS: EPSG:32749


In [6]:
#list the raster (conditioning factor) files
tif_files = sorted(DATA_RAW.rglob("*.tif"))
print(f"{len(tif_files)} raster files found:")
for t in tif_files:
    print(" ", t.name)

11 raster files found:
  map_aspect.tif
  map_distriv.tif
  map_distroad.tif
  map_elev.tif
  map_geologi.tif
  map_landuse.tif
  map_planc.tif
  map_profc.tif
  map_slope.tif
  map_spi.tif
  map_twi.tif


In [7]:
for f in sorted(DATA_RAW.rglob("*")):
    print(f.relative_to(DATA_RAW))

Landslide Controlling Factors
Landslide Controlling Factors\map_aspect.tfw
Landslide Controlling Factors\map_aspect.tif
Landslide Controlling Factors\map_aspect.tif.aux.xml
Landslide Controlling Factors\map_aspect.tif.xml
Landslide Controlling Factors\map_distriv.tfw
Landslide Controlling Factors\map_distriv.tif
Landslide Controlling Factors\map_distriv.tif.aux.xml
Landslide Controlling Factors\map_distriv.tif.xml
Landslide Controlling Factors\map_distroad.tfw
Landslide Controlling Factors\map_distroad.tif
Landslide Controlling Factors\map_distroad.tif.aux.xml
Landslide Controlling Factors\map_distroad.tif.xml
Landslide Controlling Factors\map_elev.tfw
Landslide Controlling Factors\map_elev.tif
Landslide Controlling Factors\map_elev.tif.aux.xml
Landslide Controlling Factors\map_elev.tif.xml
Landslide Controlling Factors\map_geologi.tfw
Landslide Controlling Factors\map_geologi.tif
Landslide Controlling Factors\map_geologi.tif.aux.xml
Landslide Controlling Factors\map_geologi.tif.vat.db

In [8]:
landslide_gpkg = DATA_RAW / "Landslide Point" / "ls_center.gpkg"
landslides = gpd.read_file(landslide_gpkg, layer="ls_center")
landslides["label"] = 1
print(f"{len(landslides)} landslide points")
print(landslides.columns.tolist())
landslides.head()

743 landslide points
['lslpts', 'X', 'Y', 'geometry', 'label']


,lslpts,X,Y,geometry,label
0,TRUE,509435.847792,9.093568e+06,POINT (509435.848 9093568.471),1
1,TRUE,514069.366439,9.093404e+06,POINT (514069.366 9093403.96),1
2,TRUE,514979.679513,9.094294e+06,POINT (514979.68 9094293.969),1
3,TRUE,518012.406006,9.088957e+06,POINT (518012.406 9088957.124),1
4,TRUE,518427.827438,9.089537e+06,POINT (518427.827 9089536.995),1


In [9]:
landslide_gpkg = DATA_RAW / "Landslide Point" / "ls_center.gpkg"
nonlandslide_gpkg = DATA_RAW / "Landslide Point" / "non_ls.gpkg"

landslides = gpd.read_file(landslide_gpkg)
non_landslides = gpd.read_file(nonlandslide_gpkg)

print("Landslide columns:", landslides.columns.tolist())
print("Non-landslide columns:", non_landslides.columns.tolist())
print(f"\n{len(landslides)} landslide points, {len(non_landslides)} non-landslide points")

Landslide columns: ['lslpts', 'X', 'Y', 'geometry']
Non-landslide columns: ['lslpts', 'X', 'Y', 'geometry']

743 landslide points, 743 non-landslide points


In [10]:
print(landslides["lslpts"].unique())
print(non_landslides["lslpts"].unique() if "lslpts" in non_landslides.columns else "no lslpts column in non_ls")

['TRUE']
['FALSE']


In [11]:
landslides["label"] = 1
non_landslides["label"] = 0

In [12]:
combined = pd.concat([landslides, non_landslides], ignore_index=True)
combined = gpd.GeoDataFrame(combined, crs=landslides.crs)
combined["source"] = "Indonesia_Cempaka_2017"

print(combined["label"].value_counts())
print("CRS:", combined.crs)  # confirms EPSG:32749 — UTM zone 49S, correct for this part of Indonesia
combined.head()

label
1    743
0    743
Name: count, dtype: int64
CRS: EPSG:32749


,lslpts,X,Y,geometry,label,source
0,TRUE,509435.847792,9.093568e+06,POINT (509435.848 9093568.471),1,Indonesia_Cempaka_2017
1,TRUE,514069.366439,9.093404e+06,POINT (514069.366 9093403.96),1,Indonesia_Cempaka_2017
2,TRUE,514979.679513,9.094294e+06,POINT (514979.68 9094293.969),1,Indonesia_Cempaka_2017
3,TRUE,518012.406006,9.088957e+06,POINT (518012.406 9088957.124),1,Indonesia_Cempaka_2017
4,TRUE,518427.827438,9.089537e+06,POINT (518427.827 9089536.995),1,Indonesia_Cempaka_2017


In [13]:
factor_files = sorted((DATA_RAW / "Landslide Controlling Factors").glob("*.tif"))
factor_files = [f for f in factor_files if not f.name.endswith((".ovr",))]  # skip overview files
print(f"{len(factor_files)} factor rasters:")
for f in factor_files:
    print(" ", f.stem)

11 factor rasters:
  map_aspect
  map_distriv
  map_distroad
  map_elev
  map_geologi
  map_landuse
  map_planc
  map_profc
  map_slope
  map_spi
  map_twi


In [14]:
for tif_path in factor_files:
    name = tif_path.stem.replace("map_", "")  # clean names: aspect, distriv, distroad, elev, etc.
    try:
        combined[name] = sample_raster_at_points(tif_path, combined)
        print(f"  {name}: sampled ({combined[name].notna().sum()}/{len(combined)} valid)")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

  aspect: sampled (1486/1486 valid)
  distriv: sampled (1486/1486 valid)
  distroad: sampled (1486/1486 valid)
  elev: sampled (1486/1486 valid)
  geologi: sampled (1486/1486 valid)
  landuse: sampled (1486/1486 valid)
  planc: sampled (1486/1486 valid)
  profc: sampled (1486/1486 valid)
  slope: sampled (1486/1486 valid)
  spi: sampled (1486/1486 valid)
  twi: sampled (1486/1486 valid)


In [15]:
out_path = DATA_PROCESSED / "training_table_indonesia.parquet"
combined.to_parquet(out_path)
print(f"Saved: {out_path}, shape: {combined.shape}")
combined.head()

Saved: ..\data\processed\training_table_indonesia.parquet, shape: (1486, 17)


,lslpts,X,Y,geometry,label,source,aspect,distriv,distroad,elev,geologi,landuse,planc,profc,slope,spi,twi
0,TRUE,509435.847792,9.093568e+06,POINT (509435.848 9093568.471),1,Indonesia_Cempaka_2017,171.100006,679.000000,10.000000,3.900000,2,10,-0.113,-0.002,2.300000,0.074,10.37
1,TRUE,514069.366439,9.093404e+06,POINT (514069.366 9093403.96),1,Indonesia_Cempaka_2017,277.600006,739.299988,28.299999,29.500000,2,8,0.011,-0.001,21.000000,0.850,4.81
2,TRUE,514979.679513,9.094294e+06,POINT (514979.68 9094293.969),1,Indonesia_Cempaka_2017,345.100006,702.099976,222.000000,135.199997,32,3,0.006,0.000,25.500000,0.330,4.84
3,TRUE,518012.406006,9.088957e+06,POINT (518012.406 9088957.124),1,Indonesia_Cempaka_2017,356.299988,376.399994,557.299988,198.899994,32,6,-0.071,-0.005,9.600000,0.642,9.49
4,TRUE,518427.827438,9.089537e+06,POINT (518427.827 9089536.995),1,Indonesia_Cempaka_2017,250.899994,70.000000,493.399994,155.399994,32,6,0.024,-0.001,31.200001,-4.206,3.88
